A notebook to compute average "partisan bias" scores by state & chamber

In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from knobs_functions import *
import warnings

warnings.filterwarnings('ignore')

Calculate the average value of "partisan bias" metrics by state & chamber

In [14]:
from typing import List, Dict, Tuple, Any
from fetch import _score_mapping

metrics: List[str] = ["disproportionality", "efficiency_gap", "geometric_seats_bias", "seats_bias", "votes_bias", "mean_median_average_district", "lopsided_outcomes", "declination"]
additions: Dict[str, str] = dict(zip(metrics, metrics))
_score_mapping.update(additions)
ensembles = ["base0", "pop_minus", "pop_plus", "distpair", "ust", "distpair_ust", "reversible", "county25", "county50", "county75", "county100"]

bias_table: Dict[Tuple[str, str], Any] = dict()

for state, chamber in state_chamber_list:
    bias_table[(state, chamber)] = dict()
    for m in metrics:
        all_values: List[float] = []
        for e in ensembles:
            arr = fetch_score_array(state, chamber, e, m)
            all_values.extend(arr)
        mean_value = np.mean(all_values)
        bias_table[(state, chamber)][m] = mean_value

bias_table


{('FL', 'congress'): {'disproportionality': 0.04996691939508816,
  'efficiency_gap': 0.03361709871408506,
  'geometric_seats_bias': 0.01727760717094168,
  'seats_bias': 0.021068178491720418,
  'votes_bias': 0.0066334806067300306,
  'mean_median_average_district': 0.02082536056982077,
  'lopsided_outcomes': 0.00498753994336338,
  'declination': 5.859324141018823},
 ('FL', 'upper'): {'disproportionality': 0.03511297324078745,
  'efficiency_gap': 0.01876313301424097,
  'geometric_seats_bias': 0.0023080741276096713,
  'seats_bias': 0.004975053068423038,
  'votes_bias': 0.0015217491897690442,
  'mean_median_average_district': 0.012433288333128784,
  'lopsided_outcomes': 0.0015819149177950808,
  'declination': 4.16173941699735},
 ('FL', 'lower'): {'disproportionality': 0.03907630761958009,
  'efficiency_gap': 0.02272660057545716,
  'geometric_seats_bias': 0.014067975308978678,
  'seats_bias': 0.016799499997727257,
  'votes_bias': 0.006425962390738139,
  'mean_median_average_district': 0.0201

Convert the dict to a pandas DataFrame and LaTex

TODO's
* Need to generate the LaTex columns as right-justified
* Except the header's which should be centered

I tweaked both by hand

In [23]:
def make_partisan_bias_table(*, latex_filename = None, rounding: int = 2):
    """ This is modeled after mean_diff_table() """

    index_list = [f'{a[0]} {a[1]}' for a in state_chamber_list]
    df = pd.DataFrame(columns = metrics, index = index_list)

    for state, chamber in state_chamber_list:
        for m in metrics:
            multiplier = 1 if m == "declination" else 100
            df.loc[f'{state} {chamber}', m] = bias_table[(state, chamber)][m] * multiplier
    df = df.applymap(pd.to_numeric)
    df = df.round(rounding)
    df_latex = df.copy()
    df_latex = df_latex.applymap(lambda x: f"{x:.2f}") # round values

    # combine the values and markings into dataframes to return and for Latex
    state_chamber_size_dict = {f'{state} {chamber}': f'{state} {num_seats_dict[(state, chamber)]}' 
                            for state, chamber in state_chamber_list}
    for state, chamber in state_chamber_list:
        for m in metrics:
            val = df.loc[f'{state} {chamber}', m]
            df_latex.loc[f'{state} {chamber}', m] = f'\\textcolor{{black}}{{ {val:.2f} }}' # TODO

    greek = {
        'alpha': 'α',
        'beta': 'β', 
        'delta': 'δ'
    }

    metrics_name_dict: Dict[str, str] = {
        "disproportionality": "PR", 
        "efficiency_gap": "EG",
        "geometric_seats_bias": greek['beta'],
        "seats_bias": greek['alpha'] + "_s",
        "votes_bias": greek['alpha'] + "_v",
        "mean_median_average_district": "mM", 
        "lopsided_outcomes": "LO", 
        "declination": greek['delta']
        }

    if latex_filename is not None:
        df_latex.rename(columns=metrics_name_dict, index=state_chamber_size_dict, inplace=True)
        df_latex.to_latex(latex_filename, escape=False)

    return df

df = make_partisan_bias_table(latex_filename='latex tables/partisan_bias_table.tex')
df


,disproportionality,efficiency_gap,geometric_seats_bias,seats_bias,votes_bias,mean_median_average_district,lopsided_outcomes,declination
FL congress,5.00,3.36,1.73,2.11,0.66,2.08,0.50,5.86
FL upper,3.51,1.88,0.23,0.50,0.15,1.24,0.16,4.16
FL lower,3.91,2.27,1.41,1.68,0.64,2.01,1.14,6.90
IL congress,-6.81,1.35,3.58,5.78,2.10,3.01,10.79,7.39
IL upper,-7.92,0.25,3.04,5.66,1.98,3.25,10.00,3.12
IL lower,-7.77,0.40,2.75,4.38,1.61,3.09,9.88,1.91
MI congress,3.61,5.49,7.95,8.29,2.26,3.58,5.12,11.83
MI upper,4.23,6.12,7.00,7.03,2.67,3.91,5.98,13.49
MI lower,4.27,6.15,6.65,6.67,2.78,4.20,6.73,14.48
NC congress,4.86,4.29,3.31,3.35,0.83,1.04,0.48,4.57
